In [1]:
import pandas as pd
import os
from pathlib import Path

# Define file paths
project_unit_file = r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise.xlsx"
transaction_file = r"D:\Dubai\Dubai_Updated_data\Transection_data\final_complete_analysis.xlsx"
output_file = r"D:\Dubai\Dubai_Updated_data\Dubai_Grand_Summary\Dubai_Grand_summary_test_1.xlsx"

# Create a dictionary to store the final merged dataframes
final_sheets = {}

# Define sheet name mappings
project_unit_sheets = {
    'City_Overall': 'City_Overall',
    'City_YoY': 'City_YoY', 
    'City_QoQ': 'City_QoQ',
    'Location_Overall': 'Location_Overall',
    'Location_YoY': 'Location_YoY',
    'Location_QoQ': 'Location_QoQ'
}

transaction_sheets = {
    'City_Overall': 'City_Overall',
    'City_YoY': 'City_YoY',
    'City_QoQ': 'City_QoQ',
    'Overall': 'Location_Overall',
    'YoY': 'Location_YoY',
    'QoQ': 'Location_QoQ'
}

# Helper function to lowercase column names and add source prefix (simpler approach)
def prepare_dataframe(df, prefix):
    if df.empty:
        return pd.DataFrame()
    
    # Lowercase column names
    df.columns = [str(col).strip().lower() for col in df.columns]
    
    # Add prefix to column names
    if prefix:
        df.columns = [f"{prefix}_{col}" for col in df.columns]
    
    return df

# Helper function to filter out rows where required columns are null
def filter_required_columns(df, required_cols):
    """Filter dataframe to keep only rows where all required columns have non-null values"""
    if df.empty:
        return df
    
    # Make a copy to avoid warnings
    df_filtered = df.copy()
    
    # Find which required columns exist in the dataframe
    existing_cols = [col for col in required_cols if col in df_filtered.columns]
    
    if existing_cols:
        # Drop rows where any of the required columns are null
        initial_rows = len(df_filtered)
        df_filtered = df_filtered.dropna(subset=existing_cols)
        dropped_rows = initial_rows - len(df_filtered)
        
        if dropped_rows > 0:
            print(f"    Filtered out {dropped_rows} rows with missing values in {existing_cols}")
    
    return df_filtered

# Load all sheets from both files
print("Loading Excel files...")
project_unit_data = pd.read_excel(project_unit_file, sheet_name=None)
transaction_data = pd.read_excel(transaction_file, sheet_name=None)

print(f"Project Unit File sheets: {list(project_unit_data.keys())}")
print(f"Transaction File sheets: {list(transaction_data.keys())}")

# Process each sheet type
for sheet_name in ['City_Overall', 'City_YoY', 'City_QoQ', 'Location_Overall', 'Location_YoY', 'Location_QoQ']:
    print(f"\nProcessing {sheet_name}...")
    
    # Get the corresponding sheet names from each file
    proj_sheet = sheet_name
    trans_sheet = None
    
    # Find the corresponding transaction sheet name
    for key, value in transaction_sheets.items():
        if value == sheet_name:
            trans_sheet = key
            break
    
    # Get dataframes
    df_proj = project_unit_data.get(proj_sheet, pd.DataFrame())
    df_trans = transaction_data.get(trans_sheet, pd.DataFrame())
    
    if df_proj.empty:
        print(f"  Warning: {proj_sheet} not found or empty in Project Unit file")
    
    if df_trans.empty:
        print(f"  Warning: {trans_sheet} not found or empty in Transaction file")
    
    # Prepare dataframes (lowercase and add prefix)
    df_proj_prep = prepare_dataframe(df_proj.copy(), "proj")
    df_trans_prep = prepare_dataframe(df_trans.copy(), "trans")
    
    # Merge based on sheet type
    if sheet_name == 'City_Overall':
        # Simply concatenate all rows horizontally
        if not df_proj_prep.empty and not df_trans_prep.empty:
            # Reset indices to align rows
            merged_df = pd.concat([df_proj_prep.reset_index(drop=True), 
                                 df_trans_prep.reset_index(drop=True)], axis=1)
        elif not df_proj_prep.empty:
            merged_df = df_proj_prep
        elif not df_trans_prep.empty:
            merged_df = df_trans_prep
        else:
            merged_df = pd.DataFrame()
    
    elif sheet_name == 'City_YoY':
        # Merge on year column
        if not df_proj_prep.empty and not df_trans_prep.empty:
            # Find year columns
            proj_year_cols = [col for col in df_proj_prep.columns if 'year' in col]
            trans_year_cols = [col for col in df_trans_prep.columns if 'year' in col]
            
            if proj_year_cols and trans_year_cols:
                proj_year_col = proj_year_cols[0]
                trans_year_col = trans_year_cols[0]
                
                # Perform outer join on year
                merged_df = pd.merge(df_proj_prep, df_trans_prep, 
                                    left_on=proj_year_col, 
                                    right_on=trans_year_col,
                                    how='outer')
                
                # Remove duplicate year column if it exists (keeping proj_year)
                if trans_year_col in merged_df.columns and trans_year_col != proj_year_col:
                    merged_df = merged_df.drop(columns=[trans_year_col])
            else:
                print(f"  Warning: Year column not found for {sheet_name}")
                # If no year column, just concatenate
                merged_df = pd.concat([df_proj_prep.reset_index(drop=True), 
                                     df_trans_prep.reset_index(drop=True)], axis=1)
        elif not df_proj_prep.empty:
            merged_df = df_proj_prep
        elif not df_trans_prep.empty:
            merged_df = df_trans_prep
        else:
            merged_df = pd.DataFrame()
    
    elif sheet_name == 'City_QoQ':
        # Merge on quarter column
        if not df_proj_prep.empty and not df_trans_prep.empty:
            # Find quarter columns
            proj_qtr_cols = [col for col in df_proj_prep.columns if any(q in col for q in ['quarter', 'qtr', 'q'])]
            trans_qtr_cols = [col for col in df_trans_prep.columns if any(q in col for q in ['quarter', 'qtr', 'q'])]
            
            # If no standard quarter columns, look for alternatives
            if not proj_qtr_cols:
                proj_qtr_cols = [col for col in df_proj_prep.columns if any(q in col for q in ['period', 'time', 'date'])]
            if not trans_qtr_cols:
                trans_qtr_cols = [col for col in df_trans_prep.columns if any(q in col for q in ['period', 'time', 'date'])]
            
            if proj_qtr_cols and trans_qtr_cols:
                proj_qtr_col = proj_qtr_cols[0]
                trans_qtr_col = trans_qtr_cols[0]
                
                # Perform outer join on quarter
                merged_df = pd.merge(df_proj_prep, df_trans_prep,
                                    left_on=proj_qtr_col,
                                    right_on=trans_qtr_col,
                                    how='outer')
                
                # Remove duplicate quarter column if it exists (keeping proj_qtr)
                if trans_qtr_col in merged_df.columns and trans_qtr_col != proj_qtr_col:
                    merged_df = merged_df.drop(columns=[trans_qtr_col])
            else:
                print(f"  Warning: Quarter column not found for {sheet_name}")
                # If no quarter column, just concatenate
                merged_df = pd.concat([df_proj_prep.reset_index(drop=True),
                                     df_trans_prep.reset_index(drop=True)], axis=1)
        elif not df_proj_prep.empty:
            merged_df = df_proj_prep
        elif not df_trans_prep.empty:
            merged_df = df_trans_prep
        else:
            merged_df = pd.DataFrame()
    
    elif sheet_name == 'Location_Overall':
        # Merge on area_name_en
        if not df_proj_prep.empty and not df_trans_prep.empty:
            # Find area_name_en columns
            proj_area_cols = [col for col in df_proj_prep.columns if 'area_name_en' in col]
            trans_area_cols = [col for col in df_trans_prep.columns if 'area_name_en' in col]
            
            if proj_area_cols and trans_area_cols:
                proj_area_col = proj_area_cols[0]
                trans_area_col = trans_area_cols[0]
                
                # Perform outer join on area_name_en
                merged_df = pd.merge(df_proj_prep, df_trans_prep,
                                    left_on=proj_area_col,
                                    right_on=trans_area_col,
                                    how='outer')
                
                # Remove duplicate area column if it exists (keeping proj_area)
                if trans_area_col in merged_df.columns and trans_area_col != proj_area_col:
                    merged_df = merged_df.drop(columns=[trans_area_col])
                
                # Filter to keep only rows with location values
                location_col = proj_area_col if proj_area_col in merged_df.columns else trans_area_col
                if location_col in merged_df.columns:
                    print(f"    Filtering {sheet_name} to keep only rows with location values")
                    merged_df = filter_required_columns(merged_df, [location_col])
            else:
                print(f"  Warning: area_name_en column not found for {sheet_name}")
                # If no area column, just concatenate
                merged_df = pd.concat([df_proj_prep.reset_index(drop=True),
                                     df_trans_prep.reset_index(drop=True)], axis=1)
        elif not df_proj_prep.empty:
            merged_df = df_proj_prep
            # Filter to keep only rows with location values
            area_cols = [col for col in merged_df.columns if 'area_name_en' in col]
            if area_cols:
                print(f"    Filtering {sheet_name} to keep only rows with location values")
                merged_df = filter_required_columns(merged_df, [area_cols[0]])
        elif not df_trans_prep.empty:
            merged_df = df_trans_prep
            # Filter to keep only rows with location values
            area_cols = [col for col in merged_df.columns if 'area_name_en' in col]
            if area_cols:
                print(f"    Filtering {sheet_name} to keep only rows with location values")
                merged_df = filter_required_columns(merged_df, [area_cols[0]])
        else:
            merged_df = pd.DataFrame()
    
    elif sheet_name == 'Location_YoY':
        # Merge on area_name_en and year
        if not df_proj_prep.empty and not df_trans_prep.empty:
            # Find columns
            proj_area_cols = [col for col in df_proj_prep.columns if 'area_name_en' in col]
            trans_area_cols = [col for col in df_trans_prep.columns if 'area_name_en' in col]
            proj_year_cols = [col for col in df_proj_prep.columns if 'year' in col]
            trans_year_cols = [col for col in df_trans_prep.columns if 'year' in col]
            
            if proj_area_cols and trans_area_cols and proj_year_cols and trans_year_cols:
                proj_area_col = proj_area_cols[0]
                trans_area_col = trans_area_cols[0]
                proj_year_col = proj_year_cols[0]
                trans_year_col = trans_year_cols[0]
                
                # Perform outer join on both columns
                merged_df = pd.merge(df_proj_prep, df_trans_prep,
                                    left_on=[proj_area_col, proj_year_col],
                                    right_on=[trans_area_col, trans_year_col],
                                    how='outer')
                
                # Remove duplicate columns if they exist (keeping proj columns)
                for col in [trans_area_col, trans_year_col]:
                    if col in merged_df.columns and col not in [proj_area_col, proj_year_col]:
                        merged_df = merged_df.drop(columns=[col])
                
                # Filter to keep only rows with location AND year values
                location_col = proj_area_col if proj_area_col in merged_df.columns else trans_area_col
                year_col = proj_year_col if proj_year_col in merged_df.columns else trans_year_col
                
                if location_col in merged_df.columns and year_col in merged_df.columns:
                    print(f"    Filtering {sheet_name} to keep only rows with both location AND year values")
                    merged_df = filter_required_columns(merged_df, [location_col, year_col])
            else:
                print(f"  Warning: Required columns not found for {sheet_name}")
                # If required columns not found, just concatenate
                merged_df = pd.concat([df_proj_prep.reset_index(drop=True),
                                     df_trans_prep.reset_index(drop=True)], axis=1)
        elif not df_proj_prep.empty:
            merged_df = df_proj_prep
            # Filter to keep only rows with location AND year values
            area_cols = [col for col in merged_df.columns if 'area_name_en' in col]
            year_cols = [col for col in merged_df.columns if 'year' in col]
            
            if area_cols and year_cols:
                print(f"    Filtering {sheet_name} to keep only rows with both location AND year values")
                merged_df = filter_required_columns(merged_df, [area_cols[0], year_cols[0]])
        elif not df_trans_prep.empty:
            merged_df = df_trans_prep
            # Filter to keep only rows with location AND year values
            area_cols = [col for col in merged_df.columns if 'area_name_en' in col]
            year_cols = [col for col in merged_df.columns if 'year' in col]
            
            if area_cols and year_cols:
                print(f"    Filtering {sheet_name} to keep only rows with both location AND year values")
                merged_df = filter_required_columns(merged_df, [area_cols[0], year_cols[0]])
        else:
            merged_df = pd.DataFrame()
    
    elif sheet_name == 'Location_QoQ':
        # Merge on area_name_en and quarter
        if not df_proj_prep.empty and not df_trans_prep.empty:
            # Find columns
            proj_area_cols = [col for col in df_proj_prep.columns if 'area_name_en' in col]
            trans_area_cols = [col for col in df_trans_prep.columns if 'area_name_en' in col]
            proj_qtr_cols = [col for col in df_proj_prep.columns if any(q in col for q in ['quarter', 'qtr', 'q'])]
            trans_qtr_cols = [col for col in df_trans_prep.columns if any(q in col for q in ['quarter', 'qtr', 'q'])]
            
            # If no standard quarter columns, look for alternatives
            if not proj_qtr_cols:
                proj_qtr_cols = [col for col in df_proj_prep.columns if any(q in col for q in ['period', 'time', 'date'])]
            if not trans_qtr_cols:
                trans_qtr_cols = [col for col in df_trans_prep.columns if any(q in col for q in ['period', 'time', 'date'])]
            
            if proj_area_cols and trans_area_cols and proj_qtr_cols and trans_qtr_cols:
                proj_area_col = proj_area_cols[0]
                trans_area_col = trans_area_cols[0]
                proj_qtr_col = proj_qtr_cols[0]
                trans_qtr_col = trans_qtr_cols[0]
                
                # Perform outer join on both columns
                merged_df = pd.merge(df_proj_prep, df_trans_prep,
                                    left_on=[proj_area_col, proj_qtr_col],
                                    right_on=[trans_area_col, trans_qtr_col],
                                    how='outer')
                
                # Remove duplicate columns if they exist (keeping proj columns)
                for col in [trans_area_col, trans_qtr_col]:
                    if col in merged_df.columns and col not in [proj_area_col, proj_qtr_col]:
                        merged_df = merged_df.drop(columns=[col])
                
                # Filter to keep only rows with location AND quarter values
                location_col = proj_area_col if proj_area_col in merged_df.columns else trans_area_col
                qtr_col = proj_qtr_col if proj_qtr_col in merged_df.columns else trans_qtr_col
                
                if location_col in merged_df.columns and qtr_col in merged_df.columns:
                    print(f"    Filtering {sheet_name} to keep only rows with both location AND quarter values")
                    merged_df = filter_required_columns(merged_df, [location_col, qtr_col])
            else:
                print(f"  Warning: Required columns not found for {sheet_name}")
                # If required columns not found, just concatenate
                merged_df = pd.concat([df_proj_prep.reset_index(drop=True),
                                     df_trans_prep.reset_index(drop=True)], axis=1)
        elif not df_proj_prep.empty:
            merged_df = df_proj_prep
            # Filter to keep only rows with location AND quarter values
            area_cols = [col for col in merged_df.columns if 'area_name_en' in col]
            qtr_cols = [col for col in merged_df.columns if any(q in col for q in ['quarter', 'qtr', 'q'])]
            
            if area_cols and qtr_cols:
                print(f"    Filtering {sheet_name} to keep only rows with both location AND quarter values")
                merged_df = filter_required_columns(merged_df, [area_cols[0], qtr_cols[0]])
        elif not df_trans_prep.empty:
            merged_df = df_trans_prep
            # Filter to keep only rows with location AND quarter values
            area_cols = [col for col in merged_df.columns if 'area_name_en' in col]
            qtr_cols = [col for col in merged_df.columns if any(q in col for q in ['quarter', 'qtr', 'q'])]
            
            if area_cols and qtr_cols:
                print(f"    Filtering {sheet_name} to keep only rows with both location AND quarter values")
                merged_df = filter_required_columns(merged_df, [area_cols[0], qtr_cols[0]])
        else:
            merged_df = pd.DataFrame()
    
    # Store the merged dataframe
    final_sheets[sheet_name] = merged_df
    print(f"  Completed: {sheet_name} - Shape: {merged_df.shape}")

# Save to new Excel file
print(f"\nSaving merged data to: {output_file}")

# Ensure output directory exists
output_dir = os.path.dirname(output_file)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Write to Excel
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for sheet_name, df in final_sheets.items():
        if not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            # Create empty sheet
            pd.DataFrame().to_excel(writer, sheet_name=sheet_name, index=False)
        
        print(f"  Saved: {sheet_name}")

print(f"\n✅ Merge completed successfully!")
print(f"Output file: {output_file}")
print(f"Total sheets created: {len(final_sheets)}")

# Show summary of each sheet
print("\n=== Sheet Summary ===")
for sheet_name, df in final_sheets.items():
    print(f"{sheet_name}: {df.shape[0]} rows, {df.shape[1]} columns")

Loading Excel files...
Project Unit File sheets: ['Sheet1']
Transaction File sheets: ['Overall', 'YoY', 'QoQ', 'City_Overall', 'City_YoY', 'City_QoQ']

Processing City_Overall...
  Completed: City_Overall - Shape: (2, 127)

Processing City_YoY...
  Completed: City_YoY - Shape: (33, 128)

Processing City_QoQ...
  Completed: City_QoQ - Shape: (118, 128)

Processing Location_Overall...
    Filtering Location_Overall to keep only rows with location values
  Completed: Location_Overall - Shape: (182, 136)

Processing Location_YoY...
    Filtering Location_YoY to keep only rows with both location AND year values
  Completed: Location_YoY - Shape: (3332, 137)

Processing Location_QoQ...
    Filtering Location_QoQ to keep only rows with both location AND quarter values
  Completed: Location_QoQ - Shape: (9953, 137)

Saving merged data to: D:\Dubai\Dubai_Updated_data\Dubai_Grand_Summary\Dubai_Grand_summary_test_1.xlsx
  Saved: City_Overall
  Saved: City_YoY
  Saved: City_QoQ
  Saved: Location_O

# Dictionary keys to columns

In [2]:
import pandas as pd
import json
import ast
import os
from io import BytesIO

def is_nested_dict(value):
    """Check if value is a nested dictionary (dict of dicts)."""
    if not isinstance(value, dict):
        return False
    
    # Check if any value in the dictionary is also a dictionary
    for v in value.values():
        if isinstance(v, dict):
            return True
    return False

def parse_cell_value(cell):
    """Safely parse cell value that might be a dictionary string."""
    if pd.isna(cell):
        return cell
    
    # If already a dictionary
    if isinstance(cell, dict):
        return cell
    
    # If string that looks like a dictionary
    if isinstance(cell, str):
        cell = cell.strip()
        if (cell.startswith('{') and cell.endswith('}')) or (cell.startswith('[') and cell.endswith(']')):
            try:
                # Try JSON first
                return json.loads(cell.replace("'", '"'))
            except:
                try:
                    # Try Python literal eval
                    return ast.literal_eval(cell)
                except:
                    return cell
    return cell

def process_excel_file(input_path, output_path=None):
    """
    Process Excel file with dictionary data.
    - Expands nested dictionaries into separate columns
    - Keeps single-level dictionaries as-is
    - Modifies the file in-place or creates new file
    """
    
    # Handle locked files (~$ prefix)
    filename = os.path.basename(input_path)
    if filename.startswith('~$'):
        print(f"Warning: File '{filename}' appears to be a locked Excel file.")
        print("Please close Excel and use the actual file without '~$' prefix.")
        return None
    
    # Read all sheets
    try:
        xl = pd.ExcelFile(input_path)
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return None
    
    # Create output file name
    if output_path is None:
        output_path = input_path.replace('.xlsx', '_processed.xlsx')
        if output_path == input_path:
            output_path = input_path.replace('.xls', '_processed.xlsx')
    
    # Process each sheet
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for sheet_name in xl.sheet_names:
            print(f"Processing sheet: {sheet_name}")
            
            # Read the sheet
            df = xl.parse(sheet_name)
            
            # Create a new DataFrame for processed data
            processed_df = pd.DataFrame()
            
            # Keep track of original columns that were processed
            processed_columns = {}
            
            # Process each column
            for col_idx, col_name in enumerate(df.columns):
                print(f"  Column: {col_name}")
                
                # Parse cell values
                parsed_col = df[col_name].apply(parse_cell_value)
                
                # Check first non-null value to determine structure
                sample_value = None
                for val in parsed_col:
                    if not pd.isna(val):
                        sample_value = val
                        break
                
                if sample_value is None or not isinstance(sample_value, dict):
                    # Not a dictionary, keep as-is
                    processed_df[col_name] = df[col_name]
                    continue
                
                # Check if it's a nested dictionary
                if is_nested_dict(sample_value):
                    print(f"    Found nested dictionary with keys: {list(sample_value.keys())}")
                    
                    # Get all possible outer keys from all rows
                    all_outer_keys = set()
                    for val in parsed_col:
                        if isinstance(val, dict):
                            all_outer_keys.update(val.keys())
                    
                    # Create new columns for each outer key
                    for outer_key in sorted(all_outer_keys):
                        new_col_name = f"{outer_key}_{col_name}"
                        
                        # Extract inner dictionaries for each row
                        new_column_data = []
                        for val in parsed_col:
                            if isinstance(val, dict) and outer_key in val:
                                inner_dict = val[outer_key]
                                # Keep as dictionary (not string)
                                new_column_data.append(inner_dict)
                            else:
                                new_column_data.append(None)
                        
                        processed_df[new_col_name] = new_column_data
                    
                    # Mark this column as processed
                    processed_columns[col_name] = list(all_outer_keys)
                else:
                    # Single-level dictionary, keep as-is
                    processed_df[col_name] = parsed_col
            
            # Save processed sheet
            processed_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  Saved with {len(processed_df.columns)} columns")
    
    print(f"\nProcessing complete! Saved to: {output_path}")
    return output_path

def process_with_streamlit(uploaded_file):
    """Process Excel file for Streamlit app."""
    # Read the file
    try:
        xl = pd.ExcelFile(uploaded_file)
    except Exception as e:
        return None, f"Error reading file: {e}"
    
    # Process each sheet
    processed_sheets = {}
    
    for sheet_name in xl.sheet_names:
        df = xl.parse(sheet_name)
        
        # Create a new DataFrame for processed data
        processed_df = pd.DataFrame()
        
        # Process each column
        for col_name in df.columns:
            # Parse cell values
            parsed_col = df[col_name].apply(parse_cell_value)
            
            # Check first non-null value
            sample_value = None
            for val in parsed_col:
                if not pd.isna(val):
                    sample_value = val
                    break
            
            if sample_value is None or not isinstance(sample_value, dict):
                # Not a dictionary, keep as-is
                processed_df[col_name] = df[col_name]
                continue
            
            # Check if nested dictionary
            if is_nested_dict(sample_value):
                # Get all outer keys
                all_outer_keys = set()
                for val in parsed_col:
                    if isinstance(val, dict):
                        all_outer_keys.update(val.keys())
                
                # Create new columns
                for outer_key in sorted(all_outer_keys):
                    new_col_name = f"{outer_key}_{col_name}"
                    
                    new_column_data = []
                    for val in parsed_col:
                        if isinstance(val, dict) and outer_key in val:
                            new_column_data.append(val[outer_key])
                        else:
                            new_column_data.append(None)
                    
                    processed_df[new_col_name] = new_column_data
            else:
                # Single-level dictionary
                processed_df[col_name] = parsed_col
        
        processed_sheets[sheet_name] = processed_df
    
    return processed_sheets, None

def create_download_buffer(processed_sheets):
    """Create Excel buffer for download."""
    output_buffer = BytesIO()
    
    with pd.ExcelWriter(output_buffer, engine='openpyxl') as writer:
        for sheet_name, df in processed_sheets.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    
    output_buffer.seek(0)
    return output_buffer

# Example usage for regular Python script
if __name__ == "__main__":
    # Input file (make sure Excel is closed!)
    input_file = r"D:\Dubai\Dubai_Updated_data\Dubai_Grand_Summary\Dubai_Grand_summary_Copy.xlsx"
    
    # Check if file exists
    if not os.path.exists(input_file):
        print(f"File not found: {input_file}")
        print("\nAvailable files in directory:")
        dir_path = os.path.dirname(input_file)
        for f in os.listdir(dir_path):
            if f.endswith(('.xlsx', '.xls')):
                print(f"  - {f}")
    else:
        # Process the file
        output_file = process_excel_file(input_file)
        
        if output_file:
            print(f"\nSuccessfully processed file!")
            print(f"Input:  {input_file}")
            print(f"Output: {output_file}")
            
            # Show preview
            xl_out = pd.ExcelFile(output_file)
            for sheet in xl_out.sheet_names[:2]:  # Show first 2 sheets
                df = xl_out.parse(sheet)
                print(f"\nPreview of '{sheet}' ({df.shape[0]} rows, {df.shape[1]} columns):")
                print("Columns:", list(df.columns)[:10])  # First 10 columns
                print("\nFirst 3 rows:")
                print(df.head(3).to_string())

Processing sheet: City_Overall
  Column: trans_city_name_en
  Column: trans_total_units_sold
  Column: trans_total_area_consumed
  Column: trans_avg_rate_per_sqft
  Column: trans_rate_50th_percentile
  Column: trans_rate_75th_percentile
  Column: trans_rate_90th_percentile
  Column: trans_total_sales_value
  Column: trans_prop_others_units_sold
  Column: trans_prop_others_area_consumed
  Column: trans_prop_others_total_sales
  Column: trans_prop_others_avg_rate
  Column: trans_prop_others_rate_50th
  Column: trans_prop_others_rate_75th
  Column: trans_prop_others_rate_90th
  Column: trans_prop_office_units_sold
  Column: trans_prop_office_area_consumed
  Column: trans_prop_office_total_sales
  Column: trans_prop_office_avg_rate
  Column: trans_prop_office_rate_50th
  Column: trans_prop_office_rate_75th
  Column: trans_prop_office_rate_90th
  Column: trans_prop_flat_units_sold
  Column: trans_prop_flat_area_consumed
  Column: trans_prop_flat_total_sales
  Column: trans_prop_flat_avg_rat

C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:133: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed_df[col_name] = parsed_col
C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:133: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed_df[col_name] = parsed_col
C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:133: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining

  Saved with 218 columns
Processing sheet: City_QoQ
  Column: trans_city_name_en
  Column: trans_quarter
  Column: trans_total_units_sold
  Column: trans_total_area_consumed
  Column: trans_avg_rate_per_sqft
  Column: trans_rate_50th_percentile
  Column: trans_rate_75th_percentile
  Column: trans_rate_90th_percentile
  Column: trans_total_sales_value
  Column: trans_prop_others_units_sold
  Column: trans_prop_others_area_consumed
  Column: trans_prop_others_total_sales
  Column: trans_prop_others_avg_rate
  Column: trans_prop_others_rate_50th
  Column: trans_prop_others_rate_75th
  Column: trans_prop_others_rate_90th
  Column: trans_prop_office_units_sold
  Column: trans_prop_office_area_consumed
  Column: trans_prop_office_total_sales
  Column: trans_prop_office_avg_rate
  Column: trans_prop_office_rate_50th
  Column: trans_prop_office_rate_75th
  Column: trans_prop_office_rate_90th
  Column: trans_prop_flat_units_sold
  Column: trans_prop_flat_area_consumed
  Column: trans_prop_flat_

C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:100: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed_df[col_name] = df[col_name]
C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:133: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed_df[col_name] = parsed_col
C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:133: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

    Found nested dictionary with keys: ['Others']
  Column: trans_bhk_price_units
    Found nested dictionary with keys: ['Commercial']
  Column: trans_bhk_price_area
    Found nested dictionary with keys: ['Commercial']
  Column: trans_bhk_price_sales
    Found nested dictionary with keys: ['Commercial']
  Saved with 218 columns
Processing sheet: Location_Overall
  Column: trans_area_name_en
  Column: trans_total_units_sold
  Column: trans_total_area_consumed
  Column: trans_avg_rate_per_sqft
  Column: trans_rate_50th_percentile
  Column: trans_rate_75th_percentile
  Column: trans_rate_90th_percentile
  Column: trans_total_sales_value
  Column: trans_prop_others_units_sold
  Column: trans_prop_others_area_consumed
  Column: trans_prop_others_total_sales
  Column: trans_prop_others_avg_rate
  Column: trans_prop_others_rate_50th
  Column: trans_prop_others_rate_75th
  Column: trans_prop_others_rate_90th
  Column: trans_prop_office_units_sold
  Column: trans_prop_office_area_consumed
  C

...

  Column: trans_price_range_units_sold
  Column: trans_price_range_area_consumed
  Column: trans_price_range_total_sales
  Column: trans_proptype_price_units
    Found nested dictionary with keys: ['Others']
  Column: trans_proptype_price_area
    Found nested dictionary with keys: ['Others']
  Column: trans_proptype_price_sales
    Found nested dictionary with keys: ['Others']
  Column: trans_bhk_price_units
    Found nested dictionary with keys: ['Commercial']


C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:100: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed_df[col_name] = df[col_name]
C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:100: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed_df[col_name] = df[col_name]
C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joi

  Column: trans_bhk_price_area
    Found nested dictionary with keys: ['Commercial']
  Column: trans_bhk_price_sales
    Found nested dictionary with keys: ['Commercial']
  Column: trans_raterange_units_sold
  Column: trans_raterange_area_consumed
  Column: trans_raterange_total_sales
  Column: trans_arearange_units_sold
  Column: trans_arearange_area_consumed


C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed_df[new_col_name] = new_column_data
C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed_df[new_col_name] = new_column_data
C:\Users\Admin\AppData\Local\Temp\ipykernel_17888\3617790788.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.

  Column: trans_arearange_total_sales
  Column: trans_pricerange_units_sold
  Column: trans_pricerange_area_consumed
  Column: trans_pricerange_total_sales
  Saved with 227 columns

Processing complete! Saved to: D:\Dubai\Dubai_Updated_data\Dubai_Grand_Summary\Dubai_Grand_summary_Copy_processed.xlsx

Successfully processed file!
Input:  D:\Dubai\Dubai_Updated_data\Dubai_Grand_Summary\Dubai_Grand_summary_Copy.xlsx
Output: D:\Dubai\Dubai_Updated_data\Dubai_Grand_Summary\Dubai_Grand_summary_Copy_processed.xlsx

Preview of 'City_Overall' (1 rows, 217 columns):
Columns: ['trans_city_name_en', 'trans_total_units_sold', 'trans_total_area_consumed', 'trans_avg_rate_per_sqft', 'trans_rate_50th_percentile', 'trans_rate_75th_percentile', 'trans_rate_90th_percentile', 'trans_total_sales_value', 'trans_prop_others_units_sold', 'trans_prop_others_area_consumed']

First 3 rows:
  trans_city_name_en  trans_total_units_sold  trans_total_area_consumed  trans_avg_rate_per_sqft  trans_rate_50th_percentile

In [13]:
import pandas as pd

In [16]:
df_project_unit = pd.read_excel(r"D:\Dubai\Dubai_Project_Unit_Summary_Analysis.xlsx", sheet_name='City_Overall')
df_transaction = pd.read_excel(r"D:\Dubai\final_complete_analysis.xlsx", sheet_name='City_Overall')

In [20]:
df_project_unit

,area_name_en,area_id,total_projects_commenced,total_completed_projects,total_under_construction,residential_projects_count,residential_projects_%,commercial_projects_count,commercial_projects_%,residential_+_commercial_projects_count,...,3_bhk_area,commercial_area,other_area,4_bhk_area,5_bhk_area,penthouse_area,gt_5_bhk_area,lt_1_bhk_area,total_units_supplied,total_area_supplied
0,Dubai City,Overall,2376,1280,1096,917,38.59,162,6.82,1297,...,99119133.62,118044977.4,83747398.04,26210576.33,8318787.82,1347613.2,2485298.6,71031.57,724700,7.908133e+08


In [21]:
df_transaction

,city_name_en,Total_Units_Sold,Total_Area_Consumed,Avg_Rate_per_Sqft,Rate_50th_Percentile,Rate_75th_Percentile,Rate_90th_Percentile,Total_Sales_Value,Prop_OTHERS_Units_Sold,Prop_OTHERS_Area_Consumed,...,BHK_Area_Sales,PriceRange_Units_Sold,PriceRange_Area_Consumed,PriceRange_Total_Sales,PropType_Price_Units,PropType_Price_Area,PropType_Price_Sales,BHK_Price_Units,BHK_Price_Area,BHK_Price_Sales
0,DUBAI,658812,1.255561e+09,1702.625391,1503.310139,2099.709678,2769.570788,1.501795e+12,44110,5.407105e+08,...,"{'2 BHK':{'<300':30869899.851400003,'300-600':...","{'<350000':17828,'350,000-700,000':104387,'700...","{'<350000':13880127.734109,'350,000-700,000':6...","{'<350000':4626475945.101601,'350,000-700,000'...","{'Others':{'<350000':702,'350,000-700,000':443...","{'Others':{'<350000':4949158.876164999,'350,00...","{'Others':{'<350000':156176689.5842,'350,000-7...","{'2 BHK':{'<350000':374,'350,000-700,000':3574...","{'2 BHK':{'<350000':306991.917589,'350,000-700...","{'2 BHK':{'<350000':62003225.214499995,'350,00..."
1,ALL CITIES TOTAL,658812,1.255561e+09,1702.625391,1503.310139,2099.709678,2769.570788,1.501795e+12,44110,5.407105e+08,...,"{'2 BHK':{'<300':30869899.851400003,'300-600':...","{'<350000':17828,'350,000-700,000':104387,'700...","{'<350000':13880127.734109,'350,000-700,000':6...","{'<350000':4626475945.101601,'350,000-700,000'...","{'Others':{'<350000':702,'350,000-700,000':443...","{'Others':{'<350000':4949158.876164999,'350,00...","{'Others':{'<350000':156176689.5842,'350,000-7...","{'2 BHK':{'<350000':374,'350,000-700,000':3574...","{'2 BHK':{'<350000':306991.917589,'350,000-700...","{'2 BHK':{'<350000':62003225.214499995,'350,00..."
